In [1]:
# ============================================================
# PHASE 7 — DATASET MASTER DIAGNOSIS
# CELL 1
#
# READ ONLY
# DOES NOT MODIFY ANY FILE
#
# Purpose:
#   1. ตรวจ dataset_master.csv
#   2. ตรวจว่า video003 หายตั้งแต่ master หรือไม่
#   3. ตรวจ transcript / normalized_text
#   4. ตรวจจำนวนข้อมูลราย video
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 80)
print("PHASE 7 — DATASET MASTER DIAGNOSIS")
print("=" * 80)

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT_DIAG = Path(r"C:\LipReadingSSL")

DATASET_MASTER_PATH = (
    PROJECT_ROOT_DIAG
    / "output"
    / "dataset"
    / "dataset_master.csv"
)

print()
print("Dataset master:")
print(DATASET_MASTER_PATH)

if not DATASET_MASTER_PATH.exists():
    raise FileNotFoundError(
        f"dataset_master.csv not found:\n"
        f"{DATASET_MASTER_PATH}"
    )

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

dataset_master_diag = pd.read_csv(
    DATASET_MASTER_PATH,
    low_memory=False
)

print()
print("=" * 80)
print("MASTER FILE OVERVIEW")
print("=" * 80)

print(
    f"Rows    : {len(dataset_master_diag):,}"
)

print(
    f"Columns : {len(dataset_master_diag.columns)}"
)

print()
print("Columns:")

for column in dataset_master_diag.columns:
    print(f" - {column}")

# ------------------------------------------------------------
# Required columns for diagnosis
# ------------------------------------------------------------

required_diag_columns = [
    "video_id",
    "clip_id",
    "start_frame",
    "end_frame",
    "num_frames",
    "transcript",
    "normalized_text",
    "split",
]

missing_diag_columns = [
    column
    for column in required_diag_columns
    if column not in dataset_master_diag.columns
]

if missing_diag_columns:
    raise RuntimeError(
        "Dataset master is missing columns:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_diag_columns
        )
    )

# ------------------------------------------------------------
# Video distribution
# ------------------------------------------------------------

print()
print("=" * 80)
print("VIDEO DISTRIBUTION")
print("=" * 80)

video_distribution_diag = (
    dataset_master_diag[
        "video_id"
    ]
    .astype(str)
    .value_counts(dropna=False)
    .sort_index()
)

print(
    video_distribution_diag.to_string()
)

# ------------------------------------------------------------
# Explicit expected videos
# ------------------------------------------------------------

EXPECTED_VIDEO_IDS_DIAG = [
    "video001",
    "video002",
    "video003",
]

print()
print("=" * 80)
print("EXPECTED VIDEO CHECK")
print("=" * 80)

for video_id in EXPECTED_VIDEO_IDS_DIAG:

    count = int(
        (
            dataset_master_diag[
                "video_id"
            ].astype(str)
            == video_id
        ).sum()
    )

    if count > 0:
        print(
            f"✅ {video_id:<10}: "
            f"{count:,} rows"
        )
    else:
        print(
            f"❌ {video_id:<10}: "
            f"0 rows"
        )

# ------------------------------------------------------------
# Transcript statistics
# ------------------------------------------------------------

print()
print("=" * 80)
print("TRANSCRIPT DIAGNOSIS")
print("=" * 80)

transcript_series_diag = (
    dataset_master_diag[
        "transcript"
    ]
)

normalized_series_diag = (
    dataset_master_diag[
        "normalized_text"
    ]
)

transcript_null_diag = int(
    transcript_series_diag.isna().sum()
)

transcript_empty_diag = int(
    transcript_series_diag
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

normalized_null_diag = int(
    normalized_series_diag.isna().sum()
)

normalized_empty_diag = int(
    normalized_series_diag
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    f"Transcript NULL       : "
    f"{transcript_null_diag:,}"
)

print(
    f"Transcript empty      : "
    f"{transcript_empty_diag:,}"
)

print(
    f"Normalized NULL       : "
    f"{normalized_null_diag:,}"
)

print(
    f"Normalized empty     : "
    f"{normalized_empty_diag:,}"
)

print(
    f"Transcript non-empty  : "
    f"{len(dataset_master_diag) - transcript_empty_diag:,}"
)

print(
    f"Normalized non-empty  : "
    f"{len(dataset_master_diag) - normalized_empty_diag:,}"
)

# ------------------------------------------------------------
# Transcript by video
# ------------------------------------------------------------

print()
print("=" * 80)
print("TRANSCRIPT BY VIDEO")
print("=" * 80)

transcript_by_video_diag = (
    dataset_master_diag
    .assign(
        _video=
            dataset_master_diag[
                "video_id"
            ].astype(str),

        _transcript_nonempty=
            dataset_master_diag[
                "transcript"
            ]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
    )
    .groupby("_video")
    .agg(
        samples=(
            "_video",
            "size"
        ),

        transcript_nonempty=(
            "_transcript_nonempty",
            "sum"
        ),
    )
)

transcript_by_video_diag[
    "transcript_empty"
] = (
    transcript_by_video_diag[
        "samples"
    ]
    -
    transcript_by_video_diag[
        "transcript_nonempty"
    ]
)

print(
    transcript_by_video_diag.to_string()
)

# ------------------------------------------------------------
# Split by video
# ------------------------------------------------------------

print()
print("=" * 80)
print("VIDEO × SPLIT")
print("=" * 80)

video_split_diag = pd.crosstab(
    dataset_master_diag[
        "video_id"
    ].astype(str),

    dataset_master_diag[
        "split"
    ].astype(str)
)

print(
    video_split_diag.to_string()
)

# ------------------------------------------------------------
# Sample rows
# ------------------------------------------------------------

print()
print("=" * 80)
print("FIRST 10 ROWS")
print("=" * 80)

sample_columns_diag = [
    "video_id",
    "clip_id",
    "start_frame",
    "end_frame",
    "num_frames",
    "transcript",
    "normalized_text",
    "split",
]

print(
    dataset_master_diag[
        sample_columns_diag
    ]
    .head(10)
    .to_string(index=False)
)

print()
print("=" * 80)
print("CELL 1 COMPLETE — READ ONLY")
print("=" * 80)

PHASE 7 — DATASET MASTER DIAGNOSIS

Dataset master:
C:\LipReadingSSL\output\dataset\dataset_master.csv

MASTER FILE OVERVIEW
Rows    : 81,853
Columns : 20

Columns:
 - dataset_id
 - video_id
 - clip_id
 - clip_path
 - frame_folder
 - start_frame
 - end_frame
 - num_frames
 - clip_length
 - stride
 - speaker_id
 - sentence_id
 - transcript
 - normalized_text
 - language
 - verified
 - clip_exists
 - frame_exists
 - has_transcript
 - split

VIDEO DISTRIBUTION
video_id
video001    42993
video002    38860

EXPECTED VIDEO CHECK
✅ video001  : 42,993 rows
✅ video002  : 38,860 rows
❌ video003  : 0 rows

TRANSCRIPT DIAGNOSIS
Transcript NULL       : 81,853
Transcript empty      : 81,853
Normalized NULL       : 81,853
Normalized empty     : 81,853
Transcript non-empty  : 0
Normalized non-empty  : 0

TRANSCRIPT BY VIDEO
          samples  transcript_nonempty  transcript_empty
_video                                                  
video001    42993                    0             42993
video002 

In [2]:
# ============================================================
# PHASE 7 — DATASET MASTER DIAGNOSIS
# CELL 2
#
# READ ONLY
# DOES NOT MODIFY ANY FILE
#
# Compare:
#   Phase 6 actual clip counts
#   vs
#   dataset_master.csv
# ============================================================

print("=" * 80)
print("PHASE 7 — PHASE 6 ↔ DATASET MASTER COMPARISON")
print("=" * 80)

# ------------------------------------------------------------
# Phase 6 expected counts
# จาก Phase 6 v4 ที่ผ่าน validation แล้ว
# ------------------------------------------------------------

PHASE6_EXPECTED_COUNTS_DIAG = {
    "video001": 40076,
    "video002": 31662,
    "video003": 26273,
}

# ------------------------------------------------------------
# Master counts
# ------------------------------------------------------------

master_video_counts_diag = (
    dataset_master_diag[
        "video_id"
    ]
    .astype(str)
    .value_counts()
    .to_dict()
)

# ------------------------------------------------------------
# Comparison
# ------------------------------------------------------------

comparison_rows_diag = []

for video_id, phase6_count in (
    PHASE6_EXPECTED_COUNTS_DIAG.items()
):

    master_count = int(
        master_video_counts_diag.get(
            video_id,
            0
        )
    )

    difference = (
        master_count
        -
        phase6_count
    )

    comparison_rows_diag.append({

        "video_id":
            video_id,

        "phase6_clips":
            phase6_count,

        "master_rows":
            master_count,

        "difference":
            difference,

        "status":
            (
                "MATCH"
                if difference == 0
                else "MISMATCH"
            ),
    })

comparison_diag = pd.DataFrame(
    comparison_rows_diag
)

print()
print(
    comparison_diag.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Total
# ------------------------------------------------------------

phase6_total_diag = sum(
    PHASE6_EXPECTED_COUNTS_DIAG.values()
)

master_total_diag = len(
    dataset_master_diag
)

print()
print("=" * 80)
print("TOTAL")
print("=" * 80)

print(
    f"Phase 6 total clips : "
    f"{phase6_total_diag:,}"
)

print(
    f"Master total rows   : "
    f"{master_total_diag:,}"
)

print(
    f"Difference          : "
    f"{master_total_diag - phase6_total_diag:,}"
)

# ------------------------------------------------------------
# Identify missing video
# ------------------------------------------------------------

missing_from_master_diag = []

for video_id, phase6_count in (
    PHASE6_EXPECTED_COUNTS_DIAG.items()
):

    master_count = int(
        master_video_counts_diag.get(
            video_id,
            0
        )
    )

    if master_count == 0:
        missing_from_master_diag.append(
            video_id
        )

print()
print("=" * 80)
print("MISSING VIDEOS")
print("=" * 80)

if missing_from_master_diag:

    for video_id in missing_from_master_diag:
        print(
            f"❌ {video_id} "
            f"is completely absent from dataset_master.csv"
        )

else:

    print(
        "✅ No expected video is completely missing."
    )

# ------------------------------------------------------------
# Extra videos
# ------------------------------------------------------------

expected_set_diag = set(
    PHASE6_EXPECTED_COUNTS_DIAG.keys()
)

actual_set_diag = set(
    master_video_counts_diag.keys()
)

extra_videos_diag = (
    actual_set_diag
    -
    expected_set_diag
)

print()
print("=" * 80)
print("EXTRA VIDEOS")
print("=" * 80)

if extra_videos_diag:

    for video_id in sorted(
        extra_videos_diag
    ):
        print(
            f"⚠️ Extra video: {video_id}"
        )

else:

    print(
        "None"
    )

print()
print("=" * 80)
print("CELL 2 COMPLETE — READ ONLY")
print("=" * 80)

PHASE 7 — PHASE 6 ↔ DATASET MASTER COMPARISON

video_id  phase6_clips  master_rows  difference   status
video001         40076        42993        2917 MISMATCH
video002         31662        38860        7198 MISMATCH
video003         26273            0      -26273 MISMATCH

TOTAL
Phase 6 total clips : 98,011
Master total rows   : 81,853
Difference          : -16,158

MISSING VIDEOS
❌ video003 is completely absent from dataset_master.csv

EXTRA VIDEOS
None

CELL 2 COMPLETE — READ ONLY


In [3]:
# ============================================================
# PHASE 7 — DATASET MASTER DIAGNOSIS
# CELL 3
#
# READ ONLY
# DOES NOT MODIFY ANY FILE
#
# Trace video003:
#   mouth metadata
#   clip metadata
#   NPZ shards
#   dataset_master.csv
# ============================================================

print("=" * 80)
print("PHASE 7 — VIDEO003 PIPELINE TRACE")
print("=" * 80)

TRACE_VIDEO_IDS_DIAG = [
    "video001",
    "video002",
    "video003",
]

trace_rows_diag = []

for video_id in TRACE_VIDEO_IDS_DIAG:

    output_dir = (
        PROJECT_ROOT_DIAG
        / "output"
        / video_id
    )

    mouth_metadata_path = (
        output_dir
        / "mouth_metadata.csv"
    )

    clip_metadata_path = (
        output_dir
        / "clip_metadata.csv"
    )

    shard_dir = (
        output_dir
        / "clip_npy"
    )

    # --------------------------------------------------------
    # Mouth metadata
    # --------------------------------------------------------

    if mouth_metadata_path.exists():

        try:

            mouth_df = pd.read_csv(
                mouth_metadata_path,
                low_memory=False
            )

            mouth_rows = len(
                mouth_df
            )

        except Exception:

            mouth_rows = -1

    else:

        mouth_rows = 0

    # --------------------------------------------------------
    # Clip metadata
    # --------------------------------------------------------

    if clip_metadata_path.exists():

        try:

            clip_df = pd.read_csv(
                clip_metadata_path,
                low_memory=False
            )

            clip_rows = len(
                clip_df
            )

        except Exception:

            clip_rows = -1

    else:

        clip_rows = 0

    # --------------------------------------------------------
    # NPZ shards
    # --------------------------------------------------------

    shard_files = sorted(
        shard_dir.glob("*.npz")
    ) if shard_dir.exists() else []

    shard_files_count = len(
        shard_files
    )

    shard_clip_count = 0

    for shard_path in shard_files:

        try:

            with np.load(
                shard_path,
                allow_pickle=False
            ) as shard:

                if "clips" in shard:

                    shard_clip_count += len(
                        shard["clips"]
                    )

        except Exception as e:

            print(
                f"⚠️ Failed reading "
                f"{shard_path.name}: "
                f"{type(e).__name__}: {e}"
            )

    # --------------------------------------------------------
    # Dataset master
    # --------------------------------------------------------

    master_rows = int(
        (
            dataset_master_diag[
                "video_id"
            ]
            .astype(str)
            ==
            video_id
        ).sum()
    )

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    trace_rows_diag.append({

        "video_id":
            video_id,

        "mouth_metadata":
            mouth_rows,

        "clip_metadata":
            clip_rows,

        "npz_files":
            shard_files_count,

        "npz_clips":
            shard_clip_count,

        "dataset_master":
            master_rows,

        "master_vs_npz":
            master_rows
            -
            shard_clip_count,
    })

trace_df_diag = pd.DataFrame(
    trace_rows_diag
)

print()
print(
    trace_df_diag.to_string(
        index=False
    )
)

print()
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

for _, row in trace_df_diag.iterrows():

    video_id = row["video_id"]

    npz_clips = int(
        row["npz_clips"]
    )

    master_rows = int(
        row["dataset_master"]
    )

    if npz_clips > 0 and master_rows == 0:

        print(
            f"❌ {video_id}: "
            f"Phase 6 data EXISTS "
            f"but dataset_master has 0 rows."
        )

    elif npz_clips > master_rows:

        print(
            f"⚠️ {video_id}: "
            f"{npz_clips - master_rows:,} "
            f"clips are absent from master."
        )

    elif npz_clips == master_rows:

        print(
            f"✅ {video_id}: "
            f"NPZ ↔ master count matches."
        )

print()
print("=" * 80)
print("CELL 3 COMPLETE — READ ONLY")
print("=" * 80)

PHASE 7 — VIDEO003 PIPELINE TRACE

video_id  mouth_metadata  clip_metadata  npz_files  npz_clips  dataset_master  master_vs_npz
video001           41058          40076         79      40076           42993           2917
video002           34794          31662         62      31662           38860           7198
video003           28276          26273         52      26273               0         -26273

INTERPRETATION
❌ video003: Phase 6 data EXISTS but dataset_master has 0 rows.

CELL 3 COMPLETE — READ ONLY


In [4]:
# ============================================================
# PHASE 7 — DATASET MASTER DIAGNOSIS
# CELL 4
#
# READ ONLY
# DOES NOT MODIFY ANY FILE
#
# Determine whether transcript disappears:
#   A) before dataset_master
#   B) during dataset_master construction
#   C) only normalized_text is missing
# ============================================================

print("=" * 80)
print("PHASE 7 — TRANSCRIPT SOURCE DIAGNOSIS")
print("=" * 80)

# ------------------------------------------------------------
# Candidate source files
# ------------------------------------------------------------

candidate_paths_diag = [
    PROJECT_ROOT_DIAG / "output" / "dataset" / "metadata.csv",
    PROJECT_ROOT_DIAG / "output" / "dataset" / "dataset_metadata.csv",
    PROJECT_ROOT_DIAG / "output" / "metadata.csv",
]

print()
print("Candidate metadata files:")

for path in candidate_paths_diag:

    print(
        f" - {path} "
        f"exists={path.exists()}"
    )

# ------------------------------------------------------------
# Search CSV files under output/dataset
# ------------------------------------------------------------

dataset_dir_diag = (
    PROJECT_ROOT_DIAG
    / "output"
    / "dataset"
)

csv_files_diag = sorted(
    dataset_dir_diag.glob("*.csv")
) if dataset_dir_diag.exists() else []

print()
print("=" * 80)
print("CSV FILES UNDER output/dataset")
print("=" * 80)

for path in csv_files_diag:

    print(
        f"{path.name:<35} "
        f"{path.stat().st_size:,} bytes"
    )

# ------------------------------------------------------------
# Inspect transcript-bearing CSVs
# ------------------------------------------------------------

transcript_sources_diag = []

for csv_path in csv_files_diag:

    # skip master itself
    if (
        csv_path.resolve()
        ==
        DATASET_MASTER_PATH.resolve()
    ):
        continue

    try:

        df = pd.read_csv(
            csv_path,
            low_memory=False,
            nrows=5
        )

    except Exception:

        continue

    columns_lower = {
        str(column).lower()
        for column in df.columns
    }

    has_transcript = (
        "transcript"
        in columns_lower
    )

    has_normalized = (
        "normalized_text"
        in columns_lower
    )

    if (
        has_transcript
        or
        has_normalized
    ):

        transcript_sources_diag.append(
            csv_path
        )

print()
print("=" * 80)
print("TRANSCRIPT-BEARING SOURCE FILES")
print("=" * 80)

if not transcript_sources_diag:

    print(
        "❌ No other CSV under "
        "output/dataset contains transcript "
        "or normalized_text."
    )

else:

    for path in transcript_sources_diag:

        print(
            f"📄 {path}"
        )

# ------------------------------------------------------------
# Inspect each source
# ------------------------------------------------------------

for csv_path in transcript_sources_diag:

    print()
    print("=" * 80)
    print(
        f"SOURCE : {csv_path.name}"
    )
    print("=" * 80)

    source_df_diag = pd.read_csv(
        csv_path,
        low_memory=False
    )

    print(
        f"Rows : "
        f"{len(source_df_diag):,}"
    )

    print(
        "Columns:"
    )

    for column in source_df_diag.columns:

        print(
            f" - {column}"
        )

    # --------------------------------------------------------
    # Video counts
    # --------------------------------------------------------

    if "video_id" in source_df_diag.columns:

        print()
        print(
            "Video distribution:"
        )

        print(
            source_df_diag[
                "video_id"
            ]
            .astype(str)
            .value_counts()
            .sort_index()
            .to_string()
        )

    # --------------------------------------------------------
    # Transcript
    # --------------------------------------------------------

    if "transcript" in source_df_diag.columns:

        transcript_nonempty = int(
            source_df_diag[
                "transcript"
            ]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .sum()
        )

        print()
        print(
            f"Transcript non-empty : "
            f"{transcript_nonempty:,}"
        )

    if "normalized_text" in source_df_diag.columns:

        normalized_nonempty = int(
            source_df_diag[
                "normalized_text"
            ]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .sum()
        )

        print(
            f"Normalized non-empty : "
            f"{normalized_nonempty:,}"
        )

print()
print("=" * 80)
print("CELL 4 COMPLETE — READ ONLY")
print("=" * 80)

PHASE 7 — TRANSCRIPT SOURCE DIAGNOSIS

Candidate metadata files:
 - C:\LipReadingSSL\output\dataset\metadata.csv exists=False
 - C:\LipReadingSSL\output\dataset\dataset_metadata.csv exists=False
 - C:\LipReadingSSL\output\metadata.csv exists=True

CSV FILES UNDER output/dataset
dataset_master.csv                  13,362,171 bytes
labels.csv                          3,437,915 bytes
split.csv                           7,726,160 bytes
test.csv                            6,340,309 bytes
train.csv                           7,022,083 bytes
val.csv                             221 bytes

TRANSCRIPT-BEARING SOURCE FILES
📄 C:\LipReadingSSL\output\dataset\labels.csv
📄 C:\LipReadingSSL\output\dataset\split.csv
📄 C:\LipReadingSSL\output\dataset\test.csv
📄 C:\LipReadingSSL\output\dataset\train.csv
📄 C:\LipReadingSSL\output\dataset\val.csv

SOURCE : labels.csv
Rows : 81,853
Columns:
 - video_id
 - clip_id
 - speaker_id
 - sentence_id
 - transcript
 - normalized_text
 - language
 - verified

Video dis

In [5]:
# ============================================================
# PHASE 7 — DATASET MASTER DIAGNOSIS
# CELL 5
#
# READ ONLY
# DOES NOT MODIFY ANY FILE
#
# Purpose:
#   Trace video001/video002/video003 and transcript source
#   from the actual Phase 7 input files.
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 80)
print("PHASE 7 — ACTUAL SOURCE TRACE")
print("=" * 80)

PROJECT_ROOT_DIAG = Path(r"C:\LipReadingSSL")

OUTPUT_DIR_DIAG = (
    PROJECT_ROOT_DIAG / "output"
)

DATASET_DIR_DIAG = (
    OUTPUT_DIR_DIAG / "dataset"
)

SOURCE_FILES_DIAG = {
    "metadata":
        OUTPUT_DIR_DIAG / "metadata.csv",

    "labels":
        DATASET_DIR_DIAG / "labels.csv",

    "split":
        DATASET_DIR_DIAG / "split.csv",

    "train":
        DATASET_DIR_DIAG / "train.csv",

    "val":
        DATASET_DIR_DIAG / "val.csv",

    "test":
        DATASET_DIR_DIAG / "test.csv",

    "master":
        DATASET_DIR_DIAG / "dataset_master.csv",
}

# ------------------------------------------------------------
# 1. File existence
# ------------------------------------------------------------

print()
print("=" * 80)
print("SOURCE FILES")
print("=" * 80)

for name, path in SOURCE_FILES_DIAG.items():

    print(
        f"{name:<10}: "
        f"{path} "
        f"exists={path.exists()}"
    )

# ------------------------------------------------------------
# 2. Load every existing CSV
# ------------------------------------------------------------

SOURCE_DF_DIAG = {}

for name, path in SOURCE_FILES_DIAG.items():

    if not path.exists():
        continue

    try:

        df = pd.read_csv(
            path,
            low_memory=False
        )

        SOURCE_DF_DIAG[name] = df

        print(
            f"Loaded {name:<10}: "
            f"{len(df):,} rows / "
            f"{len(df.columns)} columns"
        )

    except Exception as e:

        print(
            f"❌ Cannot read {name}: "
            f"{type(e).__name__}: {e}"
        )

# ------------------------------------------------------------
# 3. Inspect video columns
# ------------------------------------------------------------

print()
print("=" * 80)
print("VIDEO DISTRIBUTION BY SOURCE")
print("=" * 80)

for name, df in SOURCE_DF_DIAG.items():

    video_column = None

    for candidate in [
        "video_id",
        "video",
        "video_name",
    ]:

        if candidate in df.columns:
            video_column = candidate
            break

    if video_column is None:
        print()
        print(
            f"{name}: "
            "NO VIDEO COLUMN"
        )
        continue

    counts = (
        df[video_column]
        .fillna("<NULL>")
        .astype(str)
        .value_counts()
        .sort_index()
    )

    print()
    print(
        f"[{name}] "
        f"column={video_column}"
    )

    print(
        counts.to_string()
    )

# ------------------------------------------------------------
# 4. Explicit video001/002/003 check
# ------------------------------------------------------------

print()
print("=" * 80)
print("EXPLICIT VIDEO CHECK")
print("=" * 80)

EXPECTED_VIDEOS_DIAG = [
    "video001",
    "video002",
    "video003",
]

for name, df in SOURCE_DF_DIAG.items():

    video_column = None

    for candidate in [
        "video_id",
        "video",
        "video_name",
    ]:

        if candidate in df.columns:
            video_column = candidate
            break

    if video_column is None:
        continue

    print()
    print(f"[{name}]")

    values = (
        df[video_column]
        .fillna("")
        .astype(str)
    )

    for video_id in EXPECTED_VIDEOS_DIAG:

        count = int(
            (
                values == video_id
            ).sum()
        )

        if count > 0:
            print(
                f"  ✅ {video_id}: "
                f"{count:,}"
            )
        else:
            print(
                f"  ❌ {video_id}: "
                f"0"
            )

# ------------------------------------------------------------
# 5. Transcript / normalized_text columns
# ------------------------------------------------------------

print()
print("=" * 80)
print("TRANSCRIPT COLUMNS BY SOURCE")
print("=" * 80)

for name, df in SOURCE_DF_DIAG.items():

    transcript_columns = [
        column
        for column in df.columns
        if str(column).lower()
        in {
            "transcript",
            "normalized_text",
            "text",
            "sentence",
            "label",
        }
    ]

    print()
    print(
        f"[{name}]"
    )

    if not transcript_columns:

        print(
            "  ❌ No transcript/text-like column"
        )

        continue

    for column in transcript_columns:

        nonempty = int(
            df[column]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .sum()
        )

        print(
            f"  {column:<20} "
            f"non-empty={nonempty:,} "
            f"/ total={len(df):,}"
        )

# ------------------------------------------------------------
# 6. Transcript by video
# ------------------------------------------------------------

print()
print("=" * 80)
print("TRANSCRIPT BY VIDEO")
print("=" * 80)

for name, df in SOURCE_DF_DIAG.items():

    video_column = None

    for candidate in [
        "video_id",
        "video",
        "video_name",
    ]:

        if candidate in df.columns:
            video_column = candidate
            break

    transcript_column = None

    for candidate in [
        "transcript",
        "normalized_text",
        "text",
        "sentence",
        "label",
    ]:

        if candidate in df.columns:
            transcript_column = candidate
            break

    if (
        video_column is None
        or transcript_column is None
    ):
        continue

    print()
    print(
        f"[{name}] "
        f"video={video_column}, "
        f"text={transcript_column}"
    )

    temp = df.copy()

    temp["_video"] = (
        temp[video_column]
        .fillna("")
        .astype(str)
    )

    temp["_text_nonempty"] = (
        temp[transcript_column]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    result = (
        temp
        .groupby("_video")
        .agg(
            rows=(
                "_video",
                "size"
            ),
            text_nonempty=(
                "_text_nonempty",
                "sum"
            )
        )
    )

    result["text_empty"] = (
        result["rows"]
        -
        result["text_nonempty"]
    )

    print(
        result.to_string()
    )

# ------------------------------------------------------------
# 7. Final diagnosis
# ------------------------------------------------------------

print()
print("=" * 80)
print("DIAGNOSIS")
print("=" * 80)

master_df_diag = SOURCE_DF_DIAG.get(
    "master"
)

if master_df_diag is not None:

    master_videos = set(
        master_df_diag[
            "video_id"
        ]
        .fillna("")
        .astype(str)
        .unique()
    ) if "video_id" in master_df_diag.columns else set()

    print(
        "Master videos:",
        sorted(master_videos)
    )

    if "video003" not in master_videos:

        print(
            "❌ video003 is absent from "
            "dataset_master.csv"
        )

        print(
            "→ Next: identify the last source "
            "file that still contains video003."
        )

else:

    print(
        "❌ dataset_master.csv could not be loaded."
    )

print()
print("=" * 80)
print("CELL 5 COMPLETE — READ ONLY")
print("=" * 80)

PHASE 7 — ACTUAL SOURCE TRACE

SOURCE FILES
metadata  : C:\LipReadingSSL\output\metadata.csv exists=True
labels    : C:\LipReadingSSL\output\dataset\labels.csv exists=True
split     : C:\LipReadingSSL\output\dataset\split.csv exists=True
train     : C:\LipReadingSSL\output\dataset\train.csv exists=True
val       : C:\LipReadingSSL\output\dataset\val.csv exists=True
test      : C:\LipReadingSSL\output\dataset\test.csv exists=True
master    : C:\LipReadingSSL\output\dataset\dataset_master.csv exists=True
Loaded metadata  : 3 rows / 7 columns
Loaded labels    : 81,853 rows / 8 columns
Loaded split     : 81,853 rows / 8 columns
Loaded train     : 42,993 rows / 20 columns
Loaded val       : 0 rows / 20 columns
Loaded test      : 38,860 rows / 20 columns
Loaded master    : 81,853 rows / 20 columns

VIDEO DISTRIBUTION BY SOURCE

[metadata] column=video
video
video001    1
video002    1
video003    1

[labels] column=video_id
video_id
video001    42993
video002    38860

[split] column=video_i

In [6]:
# ============================================================
# PHASE 7 — CELL 6
# SOURCE TRACE v2
#
# READ ONLY
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 80)
print("PHASE 7 — SOURCE TRACE v2")
print("=" * 80)

ROOT = Path(r"C:\LipReadingSSL")

FILES = {
    "metadata":
        ROOT / "output" / "metadata.csv",

    "labels":
        ROOT / "output" / "dataset" / "labels.csv",

    "split":
        ROOT / "output" / "dataset" / "split.csv",

    "train":
        ROOT / "output" / "dataset" / "train.csv",

    "val":
        ROOT / "output" / "dataset" / "val.csv",

    "test":
        ROOT / "output" / "dataset" / "test.csv",

    "master":
        ROOT / "output" / "dataset" / "dataset_master.csv",
}


# ============================================================
# 1. Helper
# ============================================================

def find_column(df, candidates):

    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    return None


# ============================================================
# 2. Load
# ============================================================

dfs = {}

for name, path in FILES.items():

    if not path.exists():
        print(
            f"❌ {name}: missing"
        )
        continue

    df = pd.read_csv(
        path,
        low_memory=False
    )

    dfs[name] = df

    print(
        f"Loaded {name:<10}: "
        f"{len(df):,} rows"
    )


# ============================================================
# 3. VIDEO DISTRIBUTION
# ============================================================

print()
print("=" * 80)
print("VIDEO DISTRIBUTION")
print("=" * 80)

VIDEO_CANDIDATES = [
    "video_id",
    "video",
    "video_name",
]

for name, df in dfs.items():

    video_col = find_column(
        df,
        VIDEO_CANDIDATES
    )

    if video_col is None:

        print()
        print(
            f"[{name}] "
            "NO VIDEO COLUMN"
        )

        continue

    counts = (
        df[video_col]
        .fillna("")
        .astype(str)
        .value_counts()
        .sort_index()
    )

    print()
    print(
        f"[{name}] "
        f"column={video_col}"
    )

    for video_id in [
        "video001",
        "video002",
        "video003",
    ]:

        count = int(
            counts.get(
                video_id,
                0
            )
        )

        print(
            f"  {video_id}: "
            f"{count:,}"
        )


# ============================================================
# 4. TRANSCRIPT DISTRIBUTION
# ============================================================

print()
print("=" * 80)
print("TRANSCRIPT DISTRIBUTION")
print("=" * 80)

TEXT_CANDIDATES = [
    "transcript",
    "normalized_text",
    "text",
    "sentence",
    "label",
]

for name, df in dfs.items():

    text_col = find_column(
        df,
        TEXT_CANDIDATES
    )

    if text_col is None:

        print()
        print(
            f"[{name}] "
            "NO TEXT COLUMN"
        )

        continue

    text = (
        df[text_col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    nonempty = int(
        text.ne("").sum()
    )

    empty = int(
        text.eq("").sum()
    )

    unique_nonempty = int(
        text[text.ne("")]
        .nunique()
    )

    print()
    print(
        f"[{name}] "
        f"column={text_col}"
    )

    print(
        f"  non-empty : "
        f"{nonempty:,}"
    )

    print(
        f"  empty     : "
        f"{empty:,}"
    )

    print(
        f"  unique    : "
        f"{unique_nonempty:,}"
    )


# ============================================================
# 5. VIDEO × TRANSCRIPT
# ============================================================

print()
print("=" * 80)
print("VIDEO × TRANSCRIPT")
print("=" * 80)

for name, df in dfs.items():

    video_col = find_column(
        df,
        VIDEO_CANDIDATES
    )

    text_col = find_column(
        df,
        TEXT_CANDIDATES
    )

    if (
        video_col is None
        or text_col is None
    ):
        continue

    temp = pd.DataFrame({
        "video_id":
            df[video_col]
            .fillna("")
            .astype(str),

        "text":
            df[text_col]
            .fillna("")
            .astype(str)
            .str.strip(),
    })

    temp["has_text"] = (
        temp["text"] != ""
    )

    result = (
        temp
        .groupby("video_id")
        .agg(
            rows=(
                "video_id",
                "size"
            ),

            text_rows=(
                "has_text",
                "sum"
            ),
        )
    )

    result["empty_text"] = (
        result["rows"]
        -
        result["text_rows"]
    )

    print()
    print(
        f"[{name}]"
    )

    for video_id in [
        "video001",
        "video002",
        "video003",
    ]:

        if video_id in result.index:

            row = result.loc[
                video_id
            ]

            print(
                f"  {video_id}: "
                f"rows={int(row['rows']):,}, "
                f"text={int(row['text_rows']):,}, "
                f"empty={int(row['empty_text']):,}"
            )

        else:

            print(
                f"  {video_id}: "
                "❌ NOT FOUND"
            )


# ============================================================
# 6. LABELS — SAMPLE
# ============================================================

if "labels" in dfs:

    df = dfs["labels"]

    print()
    print("=" * 80)
    print("LABELS SAMPLE")
    print("=" * 80)

    print(
        df.head(10).to_string(
            index=False
        )
    )


# ============================================================
# 7. SPLIT — DISTRIBUTION
# ============================================================

if "split" in dfs:

    df = dfs["split"]

    split_col = find_column(
        df,
        ["split"]
    )

    video_col = find_column(
        df,
        VIDEO_CANDIDATES
    )

    print()
    print("=" * 80)
    print("SPLIT DISTRIBUTION")
    print("=" * 80)

    if split_col:

        print(
            df[split_col]
            .fillna("<NULL>")
            .astype(str)
            .value_counts()
            .sort_index()
            .to_string()
        )

    if video_col and split_col:

        table = pd.crosstab(
            df[video_col],
            df[split_col]
        )

        print()
        print(
            "Video × Split"
        )

        print(
            table.to_string()
        )


# ============================================================
# 8. MASTER VIDEO SET
# ============================================================

print()
print("=" * 80)
print("MASTER VIDEO SET")
print("=" * 80)

if "master" in dfs:

    master = dfs["master"]

    video_col = find_column(
        master,
        VIDEO_CANDIDATES
    )

    if video_col:

        videos = sorted(
            master[video_col]
            .dropna()
            .astype(str)
            .unique()
        )

        print(
            "Master videos:"
        )

        for video_id in videos:

            print(
                f"  - {video_id}"
            )


# ============================================================
# 9. FINAL DIAGNOSIS
# ============================================================

print()
print("=" * 80)
print("FINAL SOURCE DIAGNOSIS")
print("=" * 80)

print(
    "Expected videos:"
)

print(
    "  video001"
)

print(
    "  video002"
)

print(
    "  video003"
)

print()
print(
    "Expected Phase 6 clip counts:"
)

print(
    "  video001 = 40,076"
)

print(
    "  video002 = 31,662"
)

print(
    "  video003 = 26,273"
)

print()
print(
    "DO NOT MODIFY ANY FILE YET."
)

print("=" * 80)
print("CELL 6 COMPLETE — READ ONLY")
print("=" * 80)

PHASE 7 — SOURCE TRACE v2
Loaded metadata  : 3 rows
Loaded labels    : 81,853 rows
Loaded split     : 81,853 rows
Loaded train     : 42,993 rows
Loaded val       : 0 rows
Loaded test      : 38,860 rows
Loaded master    : 81,853 rows

VIDEO DISTRIBUTION

[metadata] column=video
  video001: 1
  video002: 1
  video003: 1

[labels] column=video_id
  video001: 42,993
  video002: 38,860
  video003: 0

[split] column=video_id
  video001: 42,993
  video002: 38,860
  video003: 0

[train] column=video_id
  video001: 42,993
  video002: 0
  video003: 0

[val] column=video_id
  video001: 0
  video002: 0
  video003: 0

[test] column=video_id
  video001: 0
  video002: 38,860
  video003: 0

[master] column=video_id
  video001: 42,993
  video002: 38,860
  video003: 0

TRANSCRIPT DISTRIBUTION

[metadata] NO TEXT COLUMN

[labels] column=transcript
  non-empty : 0
  empty     : 81,853
  unique    : 0

[split] column=normalized_text
  non-empty : 0
  empty     : 81,853
  unique    : 0

[train] column=trans